# 🎙️ Speech-to-Text API — Google Colab Setup & Test

**Stack:** faster-whisper · WhisperX (optional alignment) · pyannote.audio · FastAPI · Redis · RQ

> **Before you start:** Go to **Runtime → Change runtime type** and select **T4 GPU**.

## 1 · Environment Detection & GPU Verification

In [ ]:
import sys, os

# Detect Colab
IN_COLAB = "google.colab" in sys.modules
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    pass

print(f"Running in Colab : {IN_COLAB}")
print(f"Python           : {sys.version}")

# GPU check
import subprocess
result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if result.returncode == 0:
    print("\n--- nvidia-smi ---")
    # Print only the first 15 lines to keep output compact
    print("\n".join(result.stdout.splitlines()[:15]))
else:
    print("⚠️  nvidia-smi not found — make sure you selected a GPU runtime!")

# PyTorch CUDA info (torch may not be installed yet — that is fine)
try:
    import torch
    cuda_ok = torch.cuda.is_available()
    print(f"\ntorch {torch.__version__}  |  CUDA available: {cuda_ok}")
    if cuda_ok:
        print(f"GPU  : {torch.cuda.get_device_name(0)}")
        vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"VRAM : {vram_gb:.1f} GB")
        print(f"CUDA : {torch.version.cuda}")
        # Recommend compute_type
        if vram_gb >= 10:
            print("✅  Recommended compute_type: float16  (batch_size: 16)")
        else:
            print("⚠️  Low VRAM — use compute_type: int8  and batch_size: 8")
except ImportError:
    print("torch not installed yet — will be installed in the next step")

## 2 · System Dependencies

In [ ]:
%%bash
# Install system packages (quiet)
apt-get update -qq
apt-get install -y -qq ffmpeg libsndfile1 redis-server

# Start Redis (background, quiet)
redis-server --daemonize yes --loglevel warning
sleep 1
redis-cli ping && echo "✅ Redis is up" || echo "❌ Redis failed"

echo "✅ System packages installed"

## 3 · Python Packages — Correct Install Order

**Strategy for Colab (April 2026 image):**
- ✅ **Keep** Colab's pre-installed `torch 2.11+cu130`, `torchaudio` — do NOT reinstall
- ✅ **Remove `torchvision`** — it was built for CUDA 12.8 but torch is now CUDA 13.0; pyannote imports it and crashes
- ✅ **Pin `starlette<1.0.0`** — starlette 1.0 breaks `google-adk` AND `gradio` in Colab
- ✅ **Install `numpy>=2.2.2`** — pyannote-metrics requires it; whisperx v3.8.5 also needs `>=2.1`
- ✅ **Install each group separately** so a conflict in one group doesn't abort the rest
- ✅ **Install whisperx `--no-deps`** so pip cannot downgrade/replace torch
- ✅ **Sanity check without `sys.exit`** — print warnings only so the cell doesn't raise an error

In [ ]:
import subprocess, sys, importlib

def pip(*args, ignore_errors=False):
    """Run pip install quietly; print stderr only on failure."""
    cmd = [sys.executable, "-m", "pip", "install", "--quiet"] + list(args)
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0 and not ignore_errors:
        lines = (r.stdout + r.stderr).strip().splitlines()
        print("⚠️  pip warning/error:\n" + "\n".join(lines[-20:]))
    return r.returncode

# ---------------------------------------------------------------------------
print("=== Step 1: Upgrade pip / build tools ===")
pip("--upgrade", "pip", "setuptools", "wheel")

# ---------------------------------------------------------------------------
print("\n=== Step 2: Clone the repo (skip if already present) ===")
import os
REPO = "/content/speech2text"
if not os.path.isdir(REPO):
    r = subprocess.run(
        ["git", "clone", "--quiet", "--depth", "1",
         "https://github.com/Sanjay9745/whisperx-speech2text.git", REPO],
        capture_output=True, text=True
    )
    if r.returncode != 0:
        print("❌ git clone failed:", r.stderr)
    else:
        print(f"Cloned to {REPO}")
else:
    # Repo already exists — pull latest fixes
    r = subprocess.run(
        ["git", "-C", REPO, "pull", "--quiet", "--rebase"],
        capture_output=True, text=True
    )
    if r.returncode != 0:
        print(f"⚠️  git pull failed (using existing code): {r.stderr.strip()}")
    else:
        print(f"Repo already at {REPO} — pulled latest changes")

# ---------------------------------------------------------------------------
print("\n=== Step 3: Remove torchvision (CUDA version mismatch with torch) ===")
# Colab ships torchvision compiled for an older CUDA build than the current
# torch. pyannote.audio imports torchvision and crashes when they differ.
subprocess.run(
    [sys.executable, "-m", "pip", "uninstall", "-y", "torchvision"],
    capture_output=True
)
print("  torchvision removed (or was not installed)")

# ---------------------------------------------------------------------------
print("\n=== Step 4: Core ctranslate2 + faster-whisper ===")
pip("ctranslate2>=4.5.0", "faster-whisper>=1.2.0")

# ---------------------------------------------------------------------------
print("\n=== Step 5: pyannote stack ===")
# IMPORTANT: pin pyannote.audio==4.0.1 — the last release with a flexible
# torch dependency (torch>=2.0).  4.0.2+ hard-pins torch==2.8.0 exactly,
# which conflicts with Colab's torch 2.10+ and causes the ImportError:
#   "cannot import name 'is_opaque_value' from 'torch._library.opaque_object'"
# See: https://github.com/pyannote/pyannote-audio/commit/121054b6 (Nov 2025)
for pkg in [
    "numpy>=2.2.2",
    "pyannote.audio==4.0.1",   # last version with flexible torch>=2.0 dep
    "pyannote.core>=5.0.0",
    "pyannote.pipeline>=3.0.1",
    "omegaconf>=2.3.0",
]:
    rc = pip(pkg, ignore_errors=True)
    print(f"  {'✅' if rc == 0 else '⚠️ '} {pkg}")

# ---------------------------------------------------------------------------
print("\n=== Step 6: App framework packages ===")
# starlette MUST stay <1.0.0 — starlette 1.0 breaks google-adk AND gradio.
for pkg in [
    "fastapi>=0.124.1,<1.0.0",
    "uvicorn[standard]>=0.34.0,<1.0.0",
    "starlette>=0.49.1,<1.0.0",
    "python-multipart>=0.0.18",
    "httpx>=0.28.1",
    "huggingface-hub>=0.33.5,<1.0.0",
    "transformers>=4.48.0",
    "redis>=5.2.0,<6.0.0",
    "rq>=2.0.0,<3.0.0",
    "pydub>=0.25.1",
    "soundfile>=0.12.1",
    "librosa>=0.10.2",
    "silero-vad>=5.1.2",
    "pandas>=2.2.3",
    "nltk>=3.9.1",
    "pyyaml>=6.0.2",
    "aiofiles>=24.1.0",
    "loguru>=0.7.3",
]:
    rc = pip(pkg, ignore_errors=True)
    print(f"  {'✅' if rc == 0 else '⚠️ '} {pkg}")

# ---------------------------------------------------------------------------
print("\n=== Step 7: WhisperX v3.8.5 (--no-deps keeps Colab torch intact) ===")
pip("--no-deps", "git+https://github.com/m-bain/whisperX.git@v3.8.5")

# ---------------------------------------------------------------------------
print("\n=== Step 8: Runtime compatibility patches ===")
# These patches fix known issues between pyannote/speechbrain and torch 2.6+.
# They are applied to the installed package files — not our app code.

import site, pathlib, textwrap

def patch_file(path: pathlib.Path, old: str, new: str, label: str):
    """Replace `old` with `new` in `path`; print status."""
    if not path.exists():
        print(f"  ⏭️  {label}: file not found — skipping")
        return
    text = path.read_text()
    if new.strip() in text:
        print(f"  ✅  {label}: already patched")
        return
    if old.strip() not in text:
        print(f"  ℹ️  {label}: pattern not found — may already be fixed upstream")
        return
    path.write_text(text.replace(old, new))
    print(f"  ✅  {label}: patched")

sp = pathlib.Path(site.getsitepackages()[0])

# Patch A — torch.load weights_only (torch 2.6+ changed default to True).
# pyannote checkpoints contain OmegaConf objects that fail safe unpickling.
# We inject a compat shim into pyannote's __init__ so it runs at import time.
pyannote_init = sp / "pyannote" / "audio" / "__init__.py"
PATCH_A_OLD = "# torch.load weights_only shim — do not remove"
PATCH_A_NEW = textwrap.dedent("""\
    # torch.load weights_only shim — do not remove
    import torch as _torch
    from packaging.version import Version as _V
    if _V(_torch.__version__.split("+")[0]) >= _V("2.6.0"):
        _orig_load = _torch.load
        def _safe_load(*a, **kw):
            kw.setdefault("weights_only", False)
            return _orig_load(*a, **kw)
        _torch.load = _safe_load
    del _torch, _V
""")
# Only add the shim if it's not there yet; prepend so it runs first.
if pyannote_init.exists():
    text = pyannote_init.read_text()
    if "weights_only shim" not in text:
        pyannote_init.write_text(PATCH_A_NEW + "\n" + text)
        print("  ✅  Patch A: torch.load weights_only shim injected into pyannote.audio")
    else:
        print("  ✅  Patch A: torch.load shim already present")
else:
    print("  ⏭️  Patch A: pyannote.audio __init__.py not found — skipping")

# Patch B — speechbrain torchaudio.list_audio_backends() removed in 2.9+.
patch_file(
    sp / "speechbrain" / "utils" / "torch_audio_backend.py",
    old="available_backends = torchaudio.list_audio_backends()",
    new=("available_backends = torchaudio.list_audio_backends() "
         "if hasattr(torchaudio, 'list_audio_backends') else []"),
    label="Patch B: speechbrain torchaudio.list_audio_backends",
)

# ---------------------------------------------------------------------------
print("\n=== Sanity check ===")
checks = {
    "torch"         : ("torch",          lambda m: m.__version__),
    "CUDA available": ("torch",          lambda m: str(m.cuda.is_available())),
    "numpy"         : ("numpy",          lambda m: m.__version__),
    "faster-whisper": ("faster_whisper", lambda m: m.__version__),
    "whisperx"      : ("whisperx",       lambda m: getattr(m, "__version__", "installed (no __version__)")),
    "fastapi"       : ("fastapi",        lambda m: m.__version__),
    "pyannote.audio": ("pyannote.audio", lambda m: m.__version__),
    "loguru"        : ("loguru",         lambda m: m.__version__),
    "rq"            : ("rq",             lambda m: m.__version__),
}

failed = False
print()
for label, (modname, fn) in checks.items():
    try:
        m = importlib.import_module(modname)
        val = fn(m)
        print(f"  ✅  {label:<20} {val}")
    except Exception as e:
        print(f"  ❌  {label:<20} {e}")
        failed = True

print()
if failed:
    print("⚠️  Some checks failed — see above. Continue if they are non-critical.")
else:
    print("✅  All packages OK — ready to proceed!")


## 4 · Configuration — `config.yaml` & Environment Variables

**Where does each setting go?**

| Setting | Where to put it |
|---|---|
| Model size, batch size, chunk duration | `config.yaml` |
| HuggingFace token | `WHISPER_HF_TOKEN` env var (never commit to git) |
| API keys | `WHISPER_API_KEYS` env var |
| Redis URL | `REDIS_URL` env var |
| Device / compute type | `config.yaml` or `WHISPER_DEVICE` / `WHISPER_COMPUTE_TYPE` |

> All `WHISPER_*` env vars override the value in `config.yaml`.

In [ ]:
import os, torch

# ---------------------------------------------------------------------------
# Detect VRAM to auto-pick compute_type and batch_size
# ---------------------------------------------------------------------------
if torch.cuda.is_available():
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    compute_type = "float16" if vram_gb >= 10 else "int8"
    batch_size   = 16 if vram_gb >= 10 else 8
    device       = "cuda"
    print(f"GPU detected: {torch.cuda.get_device_name(0)}  ({vram_gb:.1f} GB)")
    print(f"  → compute_type = {compute_type}")
    print(f"  → batch_size   = {batch_size}")
else:
    compute_type = "int8"
    batch_size   = 4
    device       = "cpu"
    print("No GPU — falling back to CPU / int8")

# ---------------------------------------------------------------------------
# Write config.yaml  (adjust values here as needed)
# ---------------------------------------------------------------------------
config_content = f"""# Auto-generated by Colab notebook
model:
  size: large-v2           # tiny | base | small | medium | large-v2 | large-v3
  device: {device}
  compute_type: {compute_type}
  download_root: /content/models   # Absolute path so weights persist on Colab VM

performance:
  batch_size: {batch_size}
  max_workers: 1           # Keep 1 on single-GPU Colab
  chunk_duration_sec: 30
  use_vad: true
  queue_soft_limit: 10000
  queue_hard_limit: null

accuracy:
  beam_size: 5
  temperature: 0.0
  best_of: 5

diarization:
  enabled: true
  hf_token: ""             # Set WHISPER_HF_TOKEN env var instead

webhook:
  enabled: true
  timeout_sec: 15
  retry_count: 3

security:
  api_key_enabled: true
  api_keys:
    - "colab-test-key"     # Set WHISPER_API_KEYS env var to override

paths:
  upload_dir: /content/uploads
  output_dir: /content/outputs
  temp_dir: /content/temp
"""

cfg_path = "/content/speech2text/config.yaml" if os.path.exists("/content/speech2text") else "config.yaml"
with open(cfg_path, "w") as f:
    f.write(config_content)
print(f"\nconfig.yaml written to: {cfg_path}")

# ---------------------------------------------------------------------------
# Validate by parsing with PyYAML
# ---------------------------------------------------------------------------
import yaml
with open(cfg_path) as f:
    parsed = yaml.safe_load(f)
print("\nParsed config (model section):", parsed["model"])
print("Parsed config (performance) :", parsed["performance"])

## 5 · Environment Variables

Set sensitive values here. They take priority over `config.yaml`.

In [ ]:
import os

# ============================================================
# ⚠️  FILL IN YOUR REAL VALUES HERE BEFORE RUNNING
# ============================================================

# Hugging Face token — required for pyannote diarization.
# Get yours at: https://huggingface.co/settings/tokens
# Accept model terms at: https://huggingface.co/pyannote/speaker-diarization-3.1
os.environ["WHISPER_HF_TOKEN"] = "hf_YOUR_TOKEN_HERE"   # <-- replace

# API key(s) — comma-separated list of valid keys
os.environ["WHISPER_API_KEYS"] = "colab-test-key"

# Redis (already started by colab_setup.sh / the bash cell above)
os.environ["REDIS_URL"] = "redis://localhost:6379/0"

# Optional: disable diarization if you don't have an HF token yet
# os.environ["WHISPER_DIARIZATION_ENABLED"] = "false"

# Optional: disable API key check for easier local testing
# os.environ["WHISPER_API_KEY_ENABLED"] = "false"

# ---------------------------------------------------------------------------
# Validate: print current effective env vars (redact secrets)
# ---------------------------------------------------------------------------
env_vars = [
    "REDIS_URL", "WHISPER_HF_TOKEN", "WHISPER_API_KEYS",
    "WHISPER_DEVICE", "WHISPER_COMPUTE_TYPE", "WHISPER_MODEL_SIZE",
    "WHISPER_DIARIZATION_ENABLED", "WHISPER_API_KEY_ENABLED",
]
print("Environment variable status:")
for k in env_vars:
    v = os.environ.get(k, "(not set)")
    # Redact long secrets
    display = v[:8] + "…" if len(v) > 12 and "TOKEN" in k else v
    status  = "✅" if v != "(not set)" else "⬜"
    print(f"  {status} {k:<35} = {display}")

## 6 · Webhook Test Server (Cloudflare Tunnel)

We start a tiny FastAPI webhook receiver inside Colab and expose it via **Cloudflare Tunnel** (`cloudflared`) to get a public HTTPS URL that the Speech-to-Text API can POST results to.

> **No account or token needed** — `cloudflared` creates a free, temporary `*.trycloudflare.com` URL automatically.


In [ ]:
%%bash
pip install --quiet fastapi uvicorn

# Install cloudflared (Cloudflare Tunnel) — free, no account needed
if ! command -v cloudflared &>/dev/null; then
    wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
    dpkg -i cloudflared-linux-amd64.deb
    rm -f cloudflared-linux-amd64.deb
fi
echo "✅ cloudflared $(cloudflared --version 2>&1 | head -1)"


In [ ]:
import json, threading, os, subprocess, time, re
from pathlib import Path
from fastapi import FastAPI, Request
from fastapi.responses import JSONResponse
import uvicorn

WEBHOOK_LOG  = Path("/content/webhook_payloads.jsonl")
WEBHOOK_PORT = 4000

# ---------------------------------------------------------------------------
# Tiny webhook receiver FastAPI app
# ---------------------------------------------------------------------------
webhook_app = FastAPI()
received_payloads = []

@webhook_app.post("/webhook")
async def receive_webhook(request: Request):
    payload = await request.json()
    received_payloads.append(payload)
    with open(WEBHOOK_LOG, "a") as f:
        f.write(json.dumps(payload) + "\n")
    job_id = payload.get("job_id", "?")
    status = payload.get("status", "?")
    print(f"📩 Webhook received — job_id={job_id}  status={status}")
    if status == "completed":
        text = payload.get("result", {}).get("text", "")
        print(f"   text preview: {text[:120]}")
    return JSONResponse({"received": True})

@webhook_app.get("/health")
async def health():
    return {"status": "ok", "received": len(received_payloads)}

# ---------------------------------------------------------------------------
# Start webhook server in background thread
# ---------------------------------------------------------------------------
def _run_server():
    uvicorn.run(webhook_app, host="0.0.0.0", port=WEBHOOK_PORT, log_level="warning")

t = threading.Thread(target=_run_server, daemon=True)
t.start()
time.sleep(1)   # let uvicorn bind

# ---------------------------------------------------------------------------
# Open Cloudflare Tunnel — free, no account or token required
# ---------------------------------------------------------------------------
cf_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", f"http://localhost:{WEBHOOK_PORT}"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

# Wait up to 20 s for the public trycloudflare.com URL to appear in the output
WEBHOOK_PUBLIC_URL = None
deadline = time.time() + 20
while time.time() < deadline:
    line = cf_proc.stdout.readline()
    if not line:
        time.sleep(0.1)
        continue
    # cloudflared prints the URL on a line like:
    #   ... | INF | https://xxxx.trycloudflare.com
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if m:
        WEBHOOK_PUBLIC_URL = m.group(0) + "/webhook"
        break

if WEBHOOK_PUBLIC_URL:
    os.environ["WEBHOOK_TEST_URL"] = WEBHOOK_PUBLIC_URL
    print(f"✅ Webhook server running")
    print(f"   Local  : http://localhost:{WEBHOOK_PORT}/webhook")
    print(f"   Public : {WEBHOOK_PUBLIC_URL}")
    print(f"\n   Use this URL as webhook_url when submitting jobs.")
else:
    print("⚠️  Could not parse cloudflared URL within 20 s.")
    print("   Check cloudflared output below, then copy the URL manually:")
    # Print what we have so far
    for _ in range(10):
        line = cf_proc.stdout.readline()
        if line:
            print("  ", line.rstrip())


## 7 · Start the API Server & Worker

In [ ]:
import subprocess, time, os, sys, threading, re
from collections import deque

# Resolve repo root
REPO = "/content/speech2text" if os.path.exists("/content/speech2text") else os.getcwd()
print(f"Repo root: {REPO}")

env = os.environ.copy()
env["PYTHONPATH"] = REPO
PYTHON = sys.executable   # always use the current kernel's interpreter

# rq has no __main__.py so `python -m rq` fails.
# Use the rq binary that lives next to the Python executable instead.
import shutil
RQ_BIN = os.path.join(os.path.dirname(PYTHON), "rq")
if not os.path.exists(RQ_BIN):
    RQ_BIN = shutil.which("rq") or RQ_BIN
print(f"rq binary   : {RQ_BIN}")

# ---------------------------------------------------------------------------
# Log buffers + helpers — defined FIRST so they are always available
# ---------------------------------------------------------------------------
api_log    = deque(maxlen=500)
worker_log = deque(maxlen=500)
cf_api_log = deque(maxlen=200)

def show_api_log(n=40):
    lines = list(api_log)[-n:]
    print("\n".join(lines) if lines else "(no API log yet)")

def show_worker_log(n=40):
    lines = list(worker_log)[-n:]
    print("\n".join(lines) if lines else "(no worker log yet)")

def _drain(stream, buf: deque):
    """Continuously read a subprocess pipe into a deque so it never blocks."""
    for raw in iter(stream.readline, b""):
        line = raw.decode(errors="replace").rstrip()
        buf.append(line)
    stream.close()

def _drain_text(stream, buf: deque):
    """Same as _drain but for text-mode streams (cloudflared)."""
    for line in iter(stream.readline, ""):
        buf.append(line.rstrip())
    stream.close()

# ---------------------------------------------------------------------------
# Start FastAPI (uvicorn) in background
# ---------------------------------------------------------------------------
api_proc = subprocess.Popen(
    [PYTHON, "-m", "uvicorn", "app.main:app",
     "--host", "0.0.0.0", "--port", "8000",
     "--log-level", "info"],
    cwd=REPO, env=env,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
)
threading.Thread(target=_drain, args=(api_proc.stdout, api_log), daemon=True).start()
print(f"✅ API server started (PID {api_proc.pid}) on http://localhost:8000")

# ---------------------------------------------------------------------------
# Start RQ worker in background  (use rq binary, not python -m rq)
# Note: rq CLI uses --logging_level (underscore), not --loglevel
# ---------------------------------------------------------------------------
worker_proc = subprocess.Popen(
    [RQ_BIN, "worker", "transcription",
     "--url", os.environ.get("REDIS_URL", "redis://localhost:6379/0"),
     "--logging_level", "INFO"],
    cwd=REPO, env=env,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
)
threading.Thread(target=_drain, args=(worker_proc.stdout, worker_log), daemon=True).start()
print(f"✅ RQ worker started (PID {worker_proc.pid})")

# ---------------------------------------------------------------------------
# Wait for API to be ready (up to 30 s)
# ---------------------------------------------------------------------------
import urllib.request, json as _json

print("\nWaiting for API to be ready ", end="", flush=True)
api_ready = False
for _ in range(30):
    time.sleep(1)
    print(".", end="", flush=True)
    try:
        with urllib.request.urlopen("http://localhost:8000/health", timeout=2) as r:
            body = _json.loads(r.read())
        api_ready = True
        break
    except Exception:
        pass

print()
if api_ready:
    print(f"🟢 API health check: {body}")
else:
    print("🔴 API did not start within 30 s — last API logs:")
    show_api_log(30)

# ---------------------------------------------------------------------------
# Open Cloudflare Tunnel for the API (port 8000) — public HTTPS URL
# ---------------------------------------------------------------------------
API_PUBLIC_URL = None
if api_ready:
    cf_api_proc = subprocess.Popen(
        ["cloudflared", "tunnel", "--url", "http://localhost:8000"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    threading.Thread(target=_drain_text, args=(cf_api_proc.stdout, cf_api_log), daemon=True).start()

    # Wait up to 20 s for the trycloudflare.com URL
    deadline = time.time() + 20
    while time.time() < deadline:
        for line in list(cf_api_log):
            m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
            if m:
                API_PUBLIC_URL = m.group(0)
                break
        if API_PUBLIC_URL:
            break
        time.sleep(0.5)

    if API_PUBLIC_URL:
        os.environ["API_PUBLIC_URL"] = API_PUBLIC_URL
        print(f"\n🌐 API public URL  : {API_PUBLIC_URL}")
        print(f"   Health check   : {API_PUBLIC_URL}/health")
        print(f"   Docs (Swagger) : {API_PUBLIC_URL}/docs")
        print(f"   Submit job     : POST {API_PUBLIC_URL}/transcribe")
    else:
        print("\n⚠️  Could not get public API URL within 20 s — check cf_api_log")

# ---------------------------------------------------------------------------
# Check worker is alive — wait 5 s to let it either start or crash
# ---------------------------------------------------------------------------
time.sleep(5)
if worker_proc.poll() is None:
    print(f"\n🟢 RQ worker is running (PID {worker_proc.pid})")
    print("\n--- Worker startup log (last 20 lines) ---")
    show_worker_log(20)
else:
    print(f"\n🔴 Worker crashed immediately (exit code {worker_proc.returncode})")
    print("--- Full worker crash log ---")
    show_worker_log(60)
    print("\n⚠️  Fix the error above, then re-run this cell.")

print("\n✅ Call show_api_log() or show_worker_log() at any time to see recent output.")
if API_PUBLIC_URL:
    print(f"✅ API public URL stored in os.environ['API_PUBLIC_URL'] = {API_PUBLIC_URL}")


## 8 · End-to-End Test

In [ ]:
# ── Worker crash diagnostics ─────────────────────────────────────────────────
# Run this cell any time to check whether the worker is alive and why it may
# have crashed.  Safe to run repeatedly without restarting anything.
# ─────────────────────────────────────────────────────────────────────────────
import subprocess, sys, os, shutil

alive = worker_proc.poll() is None
print(f"Worker PID  : {worker_proc.pid}")
print(f"Worker alive: {'✅ YES' if alive else f'🔴 NO  (exit code {worker_proc.returncode})'}")
print()
print("=== Full worker log ===")
show_worker_log(80)

if not alive:
    print("\n=== Quick import smoke-test (runs in this kernel) ===")
    # Try importing the worker module directly so any ImportError is visible here
    REPO = "/content/speech2text" if os.path.exists("/content/speech2text") else os.getcwd()
    if REPO not in sys.path:
        sys.path.insert(0, REPO)
    try:
        import app.worker as _w
        print("  ✅ app.worker imported OK")
    except Exception as _e:
        print(f"  ❌ app.worker import failed:\n     {_e}")

    print("\n=== rq binary check ===")
    RQ_BIN = os.path.join(os.path.dirname(sys.executable), "rq")
    found  = shutil.which("rq")
    print(f"  Expected rq binary : {RQ_BIN}  (exists={os.path.exists(RQ_BIN)})")
    print(f"  shutil.which('rq') : {found}")
    # Use whichever exists
    rq_path = RQ_BIN if os.path.exists(RQ_BIN) else (found or RQ_BIN)
    r = subprocess.run([rq_path, "worker", "--help"], capture_output=True, text=True)
    print(r.stdout[:300] or r.stderr[:300])


In [ ]:
import requests, time, json, os

API_BASE    = "http://localhost:8000"
API_KEY     = os.environ.get("WHISPER_API_KEYS", "colab-test-key").split(",")[0].strip()
WEBHOOK_URL = os.environ.get("WEBHOOK_TEST_URL", "")
AUDIO_URL   = "https://download.samplelib.com/mp3/sample-3s.mp3"

HEADERS = {"x-api-key": API_KEY}

# ---------------------------------------------------------------------------
# 1. Verify worker is alive before submitting
# ---------------------------------------------------------------------------
if worker_proc.poll() is not None:
    print(f"🔴 Worker has already exited (code {worker_proc.returncode})!")
    print("   Last worker logs:")
    show_worker_log(30)
    raise RuntimeError("Worker is not running — fix the error above and restart cell 7")

print(f"✅ Worker alive (PID {worker_proc.pid})")

# ---------------------------------------------------------------------------
# 2. Submit job
# ---------------------------------------------------------------------------
print("\n=== Submitting transcription job ===")
resp = requests.post(
    f"{API_BASE}/transcribe",
    data={
        "audio_url"  : AUDIO_URL,
        "metadata"   : json.dumps({"source": "colab_notebook_test"}),
        "webhook_url": WEBHOOK_URL,
    },
    headers=HEADERS,
    timeout=30,
)
resp.raise_for_status()
job = resp.json()
job_id = job["job_id"]
print(f"Job submitted — id: {job_id}  status: {job['status']}")

# ---------------------------------------------------------------------------
# 3. Poll status until done — show worker log snapshot every 30 s
# ---------------------------------------------------------------------------
print("\n=== Polling status (worker logs shown every 30 s) ===")
MAX_WAIT    = 600   # seconds — model download on first run can take ~3 min
deadline    = time.time() + MAX_WAIT
last_status = ""
last_log_t  = time.time()
status      = "queued"

while time.time() < deadline:
    r = requests.get(f"{API_BASE}/status/{job_id}", headers=HEADERS, timeout=10)
    r.raise_for_status()
    d        = r.json()
    status   = d["status"]
    progress = d.get("progress", 0)

    label = f"status={status:<12}  progress={progress}%"
    if label != last_status:
        print(f"  {label}")
        last_status = label

    # Print worker log snippet every 30 s so you can see what it's doing
    if time.time() - last_log_t >= 30:
        recent = list(worker_log)[-8:]
        if recent:
            print("  [worker] " + " | ".join(recent[-3:]))
        last_log_t = time.time()

    if status == "completed":
        break
    if status == "failed":
        print(f"❌ Job failed: {d.get('error')}")
        print("\n--- Worker log (last 40 lines) ---")
        show_worker_log(40)
        break

    time.sleep(3)
else:
    print(f"\n⏱️  Timed out after {MAX_WAIT}s")
    print("\n--- Worker log (last 40 lines) — check for errors / model download progress ---")
    show_worker_log(40)

# ---------------------------------------------------------------------------
# 4. Fetch result
# ---------------------------------------------------------------------------
if status == "completed":
    print("\n=== Fetching result ===")
    r = requests.get(f"{API_BASE}/result/{job_id}", headers=HEADERS, timeout=30)
    r.raise_for_status()
    result = r.json().get("result", {})

    print(f"  language : {result.get('language', '?')}")
    print(f"  segments : {len(result.get('segments', []))}")
    print(f"  words    : {len(result.get('words', []))}")
    print(f"\n  Full text:\n  {result.get('text', '(empty)')}")

# ---------------------------------------------------------------------------
# 5. Show webhook payloads received
# ---------------------------------------------------------------------------
print("\n=== Webhook payloads received ===")
log_path = "/content/webhook_payloads.jsonl"
if os.path.exists(log_path):
    with open(log_path) as f:
        lines = f.readlines()
    print(f"  {len(lines)} webhook(s) received")
    for i, line in enumerate(lines, 1):
        p = json.loads(line)
        print(f"  [{i}] job_id={p.get('job_id')}  status={p.get('status')}")
else:
    if WEBHOOK_URL:
        print("  No log file yet — webhook may still be in transit")
    else:
        print("  (no webhook URL was set)")


## 9 · Output Browser

Browse every completed job saved to `/content/outputs/`.  
Run the next cell any time — it works independently of the test cell above.


In [ ]:
"""
Output Browser — lists every completed job and lets you inspect any result.

Files are stored as:  /content/outputs/<job_id>.json
Each file contains:   text, language, segments, words, speakers, metadata
"""
import json, os
from pathlib import Path
from datetime import datetime

OUTPUT_DIR = Path("/content/outputs")

# ── Collect all result files ────────────────────────────────────────────────
files = sorted(OUTPUT_DIR.glob("*.json"), key=lambda p: p.stat().st_mtime, reverse=True)

if not files:
    print(f"No output files found in {OUTPUT_DIR}")
    print("Run the End-to-End Test cell first to generate results.")
else:
    print(f"{'#':<4} {'Job ID':<38} {'Lang':<6} {'Segs':>5} {'Words':>6}  {'Modified':<20}  Preview")
    print("─" * 110)
    for i, fp in enumerate(files, 1):
        with open(fp) as f:
            d = json.load(f)
        mtime = datetime.fromtimestamp(fp.stat().st_mtime).strftime("%Y-%m-%d %H:%M:%S")
        lang  = d.get("language", "?")
        segs  = len(d.get("segments", []))
        words = len(d.get("words", []))
        text  = d.get("text", "").replace("\n", " ").strip()
        preview = (text[:60] + "…") if len(text) > 60 else (text or "(empty)")
        print(f"{i:<4} {fp.stem:<38} {lang:<6} {segs:>5} {words:>6}  {mtime:<20}  {preview}")

    # ── Detailed view ───────────────────────────────────────────────────────
    print(f"\n{'═' * 110}")
    SHOW_JOB = None   # ← set to a job_id string to inspect one job, or None for the latest

    target = OUTPUT_DIR / f"{SHOW_JOB}.json" if SHOW_JOB else files[0]
    print(f"\n📄 Detailed view: {target.name}\n")

    with open(target) as f:
        result = json.load(f)

    print(f"  Language    : {result.get('language', '?')}  "
          f"(p={result.get('language_probability', '?')})")
    print(f"  Segments    : {len(result.get('segments', []))}")
    print(f"  Words       : {len(result.get('words', []))}")
    meta = result.get('metadata', {})
    if meta:
        print(f"  Metadata    : {meta}")
    speakers = result.get('speakers', [])
    if speakers:
        print(f"  Speakers    : {len(speakers)} turn(s)")

    print(f"\n  ── Full transcript ──")
    print(f"  {result.get('text', '(empty)')}\n")

    segs = result.get("segments", [])
    if segs:
        print(f"  ── Segments ({len(segs)}) ──")
        for seg in segs:
            spk = f"  [{seg['speaker']}]" if seg.get('speaker') else ""
            print(f"  [{seg['start']:6.2f}s → {seg['end']:6.2f}s]{spk}  {seg['text']}")

    words = result.get("words", [])
    if words:
        print(f"\n  ── Word-level timestamps (first 20) ──")
        for w in words[:20]:
            print(f"    {w['word']:<20} {w['start']:6.3f}s → {w['end']:6.3f}s  "
                  f"(conf={w.get('probability', w.get('score', '?'))})")
        if len(words) > 20:
            print(f"    … {len(words) - 20} more words")

    print(f"\n  ── Raw JSON path ──\n  {target}")


## 10 · Cleanup (optional)

Run this cell to stop the API and worker processes when you are done.


In [ ]:
import os

# Stop API server
try:
    api_proc.terminate()
    print(f"API server stopped (PID {api_proc.pid})")
except Exception as e:
    print(f"API stop: {e}")

# Stop RQ worker
try:
    worker_proc.terminate()
    print(f"RQ worker stopped (PID {worker_proc.pid})")
except Exception as e:
    print(f"Worker stop: {e}")

# Stop webhook Cloudflare tunnel
try:
    cf_proc.terminate()
    print("Webhook cloudflared tunnel stopped")
except Exception as e:
    print(f"Webhook cloudflared stop: {e}")

# Stop API Cloudflare tunnel
try:
    cf_api_proc.terminate()
    print("API cloudflared tunnel stopped")
except Exception as e:
    print(f"API cloudflared stop: {e}")

# Kill any leftover processes
os.system("pkill -f 'uvicorn app.main'")
os.system("pkill -f 'rq worker'")
os.system("pkill -f cloudflared")

print("Done.")
